# Task 16 — Neural Networks with Keras & Training Deep Networks
**Complete phase-by-phase learning notebook**

This notebook follows the supplied `task16-readme.md`: Perceptrons → MLPs/backpropagation → task-specific output layers → Keras Sequential and Functional APIs → callbacks/tuning → stable deep training → optimizers/schedules → regularization/capstone.

**How to use**
- Run from top to bottom in Jupyter.
- The notebook downloads Fashion MNIST and California Housing the first time those datasets are used; internet access may be required.
- Some experiments train multiple models and may take time. Set `QUICK_RUN = True` for a fast smoke-test; set it to `False` for fuller experiments.
- Results are generated when you run the notebook; no accuracy or score is fabricated in advance.
- TensorBoard, SciKeras, and Keras Tuner are optional add-ons and have fallback paths or clearly marked install cells.

In [ ]:
# If needed, run this in a separate notebook cell, then restart the kernel:
# %pip install -U tensorflow scikit-learn matplotlib pandas scipy scikeras tensorboard

## 0. Imports, reproducibility, and run settings

In [ ]:
import os, time, math, tempfile, datetime, warnings
from functools import partial
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_iris, fetch_california_housing
from sklearn.linear_model import Perceptron, SGDClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
from scipy.stats import reciprocal

SEED = 42
QUICK_RUN = True  # Set False for longer/full experiments
EPOCHS_MAIN = 3 if QUICK_RUN else 30
EPOCHS_EXPERIMENT = 2 if QUICK_RUN else 8
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("Quick run:", QUICK_RUN)

## Phase 1 — From biological neurons to the Perceptron

### Concepts
Biological neurons receive signals, combine them, and may activate. Artificial neural networks borrow the broad idea of weighted inputs and outputs, but they are mathematical models—not realistic simulations of brains.

A **threshold logic unit (TLU)** computes a weighted sum \(z=x^T w\) and applies a step function. Two conventions are:
- Heaviside: \(H(z)=0\) for \(z<0\), and \(1\) for \(z\geq0\).
- Sign: \(\operatorname{sign}(z)=-1,0,1\) for negative, zero, positive inputs.

A **Perceptron** is a layer of TLUs. In matrix form, \(H_{W,b}(X)=\phi(XW+b)\), where `X` has shape `(instances, features)`, `W` has shape `(features, neurons)`, and `b` has one value per neuron.

The learning rule is \(w_{ij}\leftarrow w_{ij}+\eta(y_j-\hat y_j)x_i\). It resembles Hebb's “cells that fire together wire together” idea, but includes an error term. For linearly separable data, the classic Perceptron learning algorithm converges; its solution need not be unique and its predictions are hard labels, not calibrated probabilities.

A single Perceptron cannot solve XOR because no single straight line separates the positive and negative examples. A hidden layer makes a nonlinear boundary possible.

In [ ]:
# Train a Perceptron on Iris: setosa vs. not setosa using petal length/width.
iris = load_iris()
X_iris = iris.data[:, (2, 3)]
y_iris = (iris.target == 0).astype(int)

per_clf = Perceptron(random_state=SEED, max_iter=1000, tol=1e-3)
per_clf.fit(X_iris, y_iris)
sample = np.array([[2.0, 0.5]])
print("Sample prediction (1=setosa):", per_clf.predict(sample))
print("Training accuracy:", accuracy_score(y_iris, per_clf.predict(X_iris)))

# Compare with the equivalent SGDClassifier configuration (same algorithm family).
sgd_per = SGDClassifier(loss="perceptron", learning_rate="constant",
                        eta0=1, penalty=None, max_iter=1000, tol=1e-3,
                        random_state=SEED)
sgd_per.fit(X_iris, y_iris)
print("SGDClassifier training accuracy:", accuracy_score(y_iris, sgd_per.predict(X_iris)))
print("Predictions may differ due to implementation/shuffling/stopping details.")

In [ ]:
# XOR truth table and why a single linear threshold cannot separate it.
xor_X = np.array([[0,0], [0,1], [1,0], [1,1]])
xor_y = np.array([0,1,1,0])
print(pd.DataFrame({"A": xor_X[:,0], "B": xor_X[:,1], "XOR": xor_y}))
# Geometric check: positive points (0,1),(1,0) lie on opposite corners;
# no single line can separate them from (0,0),(1,1).

**Phase 1 deliverable:** TLU definition, layer equation and shapes, Perceptron rule, convergence condition, trained classifier, and XOR argument are documented above.

## Phase 2 — MLP, backpropagation, and activations

An **MLP** contains an input layer, one or more hidden layers, and an output layer. Information moves forward (feedforward). Lower layers are closer to inputs; upper layers are closer to outputs. A network with multiple hidden layers is commonly called a deep neural network.

**Backpropagation, in order**
1. Process a mini-batch; one pass over the full training set is an epoch.
2. Forward pass: compute layer outputs and retain intermediate values.
3. Loss: compare predictions with targets.
4. Backward pass: use the chain rule to compute gradients from output to earlier layers.
5. Gradient step: update parameters to reduce loss.

Backpropagation uses **reverse-mode automatic differentiation**, efficient when there are many parameters but one scalar loss. Weights start randomly to break symmetry; if neurons start with identical weights, they receive identical gradients and learn identical features. Biases can often start at zero.

The step function is unsuitable for gradient training because its derivative is zero almost everywhere. Sigmoid is \(\sigma(z)=1/(1+e^{-z})\), tanh is \(2\sigma(2z)-1\), and ReLU is \(\max(0,z)\). ReLU is fast and does not saturate for positive inputs, but units can “die” when they stay in the negative region.

Linear layers alone collapse to a single linear transformation: if \(g(x)=5x-1\) and \(f(x)=2x+3\), then \(f(g(x))=2(5x-1)+3=10x+1\). Nonlinear activations are therefore needed between hidden layers. A sufficiently expressive network with nonlinear activations can approximate broad classes of continuous functions on compact domains (universal-approximation intuition).

In [ ]:
z = np.linspace(-6, 6, 400)
sigmoid = 1 / (1 + np.exp(-z))
tanh = np.tanh(z)
relu = np.maximum(0, z)
plt.figure(figsize=(9, 5))
plt.plot(z, sigmoid, label="Sigmoid")
plt.plot(z, tanh, label="tanh")
plt.plot(z, relu, label="ReLU")
plt.axhline(0, linewidth=0.7); plt.axvline(0, linewidth=0.7)
plt.title("Classic activation functions"); plt.xlabel("z"); plt.ylabel("activation")
plt.legend(); plt.grid(True); plt.show()
print("f(g(x)) = 2(5x - 1) + 3 = 10x + 1")

## Phase 3 — Regression and classification MLPs

### Architecture reference tables

| Setting | Regression MLP |
|---|---|
| Input neurons | One per input feature |
| Hidden layers | Often start with 1; add capacity as needed |
| Hidden neurons | Tune empirically (e.g., 10–100 as a starting range) |
| Output neurons | One per target dimension |
| Hidden activation | ReLU or a suitable alternative |
| Output activation | None by default; ReLU/softplus for positive outputs; sigmoid/tanh for bounded outputs |
| Loss | MSE by default; MAE or Huber for outlier robustness |

| Setting | Binary | Multilabel binary | Multiclass |
|---|---|---|---|
| Output neurons | 1 | 1 per label | 1 per class |
| Output activation | Sigmoid | Sigmoid per label | Softmax |
| Loss | Binary cross-entropy | Binary cross-entropy | Categorical cross-entropy |
| Label format | 0/1 | Multi-hot vector | Integer labels with sparse CE, or one-hot with categorical CE |

Scenario choices:
- House price: one linear output; MSE/Huber.
- Spam vs. ham: one sigmoid output; binary cross-entropy.
- Spam + urgent tags: two sigmoid outputs; binary cross-entropy.
- Ten mutually exclusive digit classes: ten softmax outputs; sparse categorical cross-entropy for integer labels.

In [ ]:
# Tiny demonstration: sparse labels vs one-hot labels.
integer_labels = np.array([0, 2, 1])
one_hot_labels = keras.utils.to_categorical(integer_labels, num_classes=3)
print("Integer labels:", integer_labels)
print("One-hot labels:\n", one_hot_labels)
print("Recovered integer labels:", np.argmax(one_hot_labels, axis=1))

## Phase 4 — Keras Sequential API: Fashion MNIST and California Housing

Fashion MNIST contains 28×28 grayscale images in 10 classes. We reserve the first 5,000 training examples for validation and scale pixels to `[0,1]`. The model has 300 and 100 ReLU hidden units, then a 10-unit softmax output.

Use `sparse_categorical_crossentropy` when targets are integer class IDs, `categorical_crossentropy` for one-hot targets, and `binary_crossentropy` for binary/multilabel sigmoid outputs. Test data should be used for final evaluation only, after model choices are settled.

In [ ]:
fashion_mnist = keras.datasets.fashion_mnist
(X_train_full_raw, y_train_full), (X_test_raw, y_test) = fashion_mnist.load_data()
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

X_valid = X_train_full_raw[:5000].astype("float32") / 255.0
X_train = X_train_full_raw[5000:].astype("float32") / 255.0
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]
X_test = X_test_raw.astype("float32") / 255.0

plt.imshow(X_train[0], cmap="gray")
plt.title(f"First training image: {class_names[y_train[0]]}")
plt.axis("off"); plt.show()
print("Train:", X_train.shape, "Validation:", X_valid.shape, "Test:", X_test.shape)

In [ ]:
def build_fashion_baseline():
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28)),
        keras.layers.Flatten(),
        keras.layers.Dense(300, activation="relu"),
        keras.layers.Dense(100, activation="relu"),
        keras.layers.Dense(10, activation="softmax")
    ])
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=keras.optimizers.SGD(learning_rate=0.01),
                  metrics=["accuracy"])
    return model

baseline_model = build_fashion_baseline()
baseline_model.summary()
# Dense parameter counts: 784*300+300; 300*100+100; 100*10+10.
print("Manual Dense counts:", 784*300+300, 300*100+100, 100*10+10)
baseline_history = baseline_model.fit(
    X_train, y_train, epochs=EPOCHS_MAIN,
    validation_data=(X_valid, y_valid), verbose=1
)
pd.DataFrame(baseline_history.history).plot(figsize=(9, 5), grid=True)
plt.title("Phase 4 baseline learning curves"); plt.show()

In [ ]:
# Final test evaluation for this baseline. In a production workflow, do this only after tuning.
baseline_test_loss, baseline_test_acc = baseline_model.evaluate(X_test, y_test, verbose=0)
print(f"Baseline test loss: {baseline_test_loss:.4f}")
print(f"Baseline test accuracy: {baseline_test_acc:.4f}")

probs = baseline_model.predict(X_test[:10], verbose=0)
preds = np.argmax(probs, axis=1)
print(pd.DataFrame({"true": [class_names[i] for i in y_test[:10]],
                    "predicted": [class_names[i] for i in preds],
                    "confidence": probs.max(axis=1).round(3)}))

### California Housing regression MLP

This section downloads California Housing, makes train/validation/test splits, fits `StandardScaler` on training data only, and trains a small regression MLP. The original Task 1 baseline is not included in the supplied file, so enter its test MSE below for a direct comparison.

In [ ]:
housing = fetch_california_housing()
X_h, y_h = housing.data, housing.target
X_h_train_full, X_h_test, y_h_train_full, y_h_test = train_test_split(
    X_h, y_h, test_size=0.2, random_state=SEED
)
X_h_train, X_h_valid, y_h_train, y_h_valid = train_test_split(
    X_h_train_full, y_h_train_full, test_size=0.2, random_state=SEED
)
housing_scaler = StandardScaler()
X_h_train_s = housing_scaler.fit_transform(X_h_train)
X_h_valid_s = housing_scaler.transform(X_h_valid)
X_h_test_s = housing_scaler.transform(X_h_test)

housing_model = keras.Sequential([
    keras.layers.Input(shape=(X_h_train_s.shape[1],)),
    keras.layers.Dense(30, activation="relu"),
    keras.layers.Dense(1)
])
housing_model.compile(loss="mean_squared_error", optimizer=keras.optimizers.SGD(learning_rate=0.01))
housing_history = housing_model.fit(
    X_h_train_s, y_h_train, epochs=EPOCHS_MAIN,
    validation_data=(X_h_valid_s, y_h_valid), verbose=1
)
housing_test_mse = housing_model.evaluate(X_h_test_s, y_h_test, verbose=0)
print("California Housing test MSE:", housing_test_mse)
TASK1_BASELINE_TEST_MSE = None  # Replace with the saved Task 1 baseline test MSE.
if TASK1_BASELINE_TEST_MSE is not None:
    print("Task 1 baseline test MSE:", TASK1_BASELINE_TEST_MSE)
    print("MLP minus baseline MSE:", housing_test_mse - TASK1_BASELINE_TEST_MSE)
else:
    print("Task 1 baseline MSE unavailable in supplied Task 16 document; enter it above for comparison.")

## Phase 5 — Functional API, saving, callbacks, TensorBoard, and tuning

The Functional API supports skip connections, multiple inputs, and multiple outputs. A **Wide & Deep** model concatenates raw inputs with learned hidden features: the wide path can preserve simple/hand-engineered rules, while the deep path learns combinations. Subclassing (`__init__` plus `call`) is useful for dynamic/custom computations, though its structure is less inspectable and serialization requires care.

Keras `.keras` is the recommended native format in modern Keras; the requested `.h5` format is also demonstrated. `ModelCheckpoint` saves the best validation model, `EarlyStopping` restores best weights, and TensorBoard records runs for comparison.

In [ ]:
# Functional Wide & Deep regression model using California Housing data.
input_ = keras.layers.Input(shape=(X_h_train_s.shape[1],), name="features")
hidden1 = keras.layers.Dense(30, activation="relu")(input_)
hidden2 = keras.layers.Dense(30, activation="relu")(hidden1)
concat = keras.layers.Concatenate()([input_, hidden2])
wide_deep_output = keras.layers.Dense(1, name="main_output")(concat)
wide_deep_model = keras.Model(inputs=input_, outputs=wide_deep_output)
wide_deep_model.compile(loss="mse", optimizer=keras.optimizers.Adam(learning_rate=1e-3))
wide_deep_model.summary()

In [ ]:
# Multi-input and multi-output example: wide features and deep features; auxiliary output.
n_features = X_h_train_s.shape[1]
wide_input = keras.layers.Input(shape=(n_features,), name="wide_input")
deep_input = keras.layers.Input(shape=(n_features,), name="deep_input")
deep = keras.layers.Dense(30, activation="relu")(deep_input)
deep = keras.layers.Dense(30, activation="relu")(deep)
joined = keras.layers.Concatenate()([wide_input, deep])
main_output = keras.layers.Dense(1, name="main_output")(joined)
aux_output = keras.layers.Dense(1, name="aux_output")(deep)
multi_model = keras.Model(inputs=[wide_input, deep_input], outputs=[main_output, aux_output])
multi_model.compile(optimizer="adam", loss=["mse", "mse"], loss_weights=[0.9, 0.1])
print("Multi-input/multi-output model built.")

In [ ]:
# Save and restore a Functional model.
save_path = "task16_wide_deep.keras"
wide_deep_model.save(save_path)
reloaded_model = keras.models.load_model(save_path)
print("Reloaded model:", reloaded_model.name)

In [ ]:
class ValidationLossRatio(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        tr, va = logs.get("loss"), logs.get("val_loss")
        if tr is not None and va is not None:
            print(f"Epoch {epoch+1}: val/train loss ratio = {va / max(tr, 1e-12):.3f}")

run_stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir = os.path.join("logs", "task16", run_stamp)
callbacks_model = build_fashion_baseline()
callbacks = [
    keras.callbacks.ModelCheckpoint("task16_best.keras", monitor="val_loss", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2 if QUICK_RUN else 10,
                                  restore_best_weights=True),
    keras.callbacks.TensorBoard(log_dir=log_dir),
    ValidationLossRatio()
]
callbacks_history = callbacks_model.fit(
    X_train, y_train, epochs=EPOCHS_MAIN + 2,
    validation_data=(X_valid, y_valid), callbacks=callbacks, verbose=1
)
print("TensorBoard log directory:", log_dir)
print("In a terminal, run: tensorboard --logdir logs/task16")

### Randomized hyperparameter search (optional / longer-running)

The original `keras.wrappers.scikit_learn.KerasRegressor` API is deprecated/removed in many current TensorFlow versions. This uses the modern SciKeras wrapper if installed. Install with `%pip install scikeras` and restart the kernel if needed. CV training is compute-intensive, so the default quick-run uses a small search.

In [ ]:
try:
    from scikeras.wrappers import KerasRegressor
    def build_housing_model(n_hidden=1, n_neurons=30, learning_rate=1e-3):
        model = keras.Sequential()
        model.add(keras.layers.Input(shape=(X_h_train_s.shape[1],)))
        for _ in range(n_hidden):
            model.add(keras.layers.Dense(n_neurons, activation="relu"))
        model.add(keras.layers.Dense(1))
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss="mse")
        return model

    reg = KerasRegressor(model=build_housing_model, verbose=0, epochs=3 if QUICK_RUN else 20,
                         batch_size=32, random_state=SEED)
    param_dist = {
        "model__n_hidden": [1, 2, 3],
        "model__n_neurons": [16, 30, 50],
        "model__learning_rate": reciprocal(3e-4, 3e-2),
        "batch_size": [32, 64]
    }
    search = RandomizedSearchCV(reg, param_distributions=param_dist,
                                n_iter=2 if QUICK_RUN else 10, cv=3,
                                random_state=SEED, scoring="neg_mean_squared_error")
    search.fit(X_h_train_s, y_h_train)
    print("Best parameters:", search.best_params_)
    print("Best CV negative MSE:", search.best_score_)
except ImportError:
    print("SciKeras is not installed. Run: %pip install scikeras, then rerun this cell.")

**Tuning guidelines from the supplied task:** start with one hidden layer; add capacity until overfitting begins; the same width in each layer is a reasonable baseline; early stopping and regularization allow a slightly over-sized model; learning rate is especially important; large batches can be efficient but may generalize worse; ReLU is a common hidden-layer default. Other search tools named in the task include Hyperopt, Keras Tuner, Scikit-Optimize, Spearmint, Hyperband, Hyperas/kopt/Talos, and Population-Based Training.

## Phase 6 — Initialization, activation variants, Batch Normalization, clipping, and transfer learning

**Vanishing gradients** shrink as they propagate toward earlier layers; **exploding gradients** grow and can destabilize weights. Variance-scaled initialization helps preserve signal:
- Glorot/Xavier: tanh, sigmoid, linear-ish outputs; variance scale \(1/\text{fan\_avg}\).
- He: ReLU variants; \(2/\text{fan\_in}\).
- LeCun: SELU; \(1/\text{fan\_in}\).

Dying ReLUs can be mitigated with leaky ReLU, randomized leaky ReLU (RReLU), learnable PReLU, ELU, or SELU under its self-normalizing conditions (plain MLP, standardized inputs, LeCun-normal initialization, and AlphaDropout if dropout is used).

Batch Normalization computes mini-batch mean and variance, normalizes values, then applies learned scale \(\gamma\) and shift \(\beta\). At inference, it uses moving averages from training. Gradient clipping by value clips each coordinate and can change direction; clipping by norm rescales the whole gradient vector and preserves direction.

In [ ]:
# Batch Normalization + He initialization + ELU.
def build_bn_fashion():
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28)),
        keras.layers.Flatten(),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(300, kernel_initializer="he_normal", use_bias=False),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("elu"),
        keras.layers.Dense(100, kernel_initializer="he_normal", use_bias=False),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("elu"),
        keras.layers.Dense(10, activation="softmax")
    ])
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=keras.optimizers.Adam(learning_rate=1e-3), metrics=["accuracy"])
    return model

bn_model = build_bn_fashion()
bn_history = bn_model.fit(X_train, y_train, epochs=EPOCHS_EXPERIMENT,
                          validation_data=(X_valid, y_valid), verbose=1)

In [ ]:
# Activation / initialization comparison table based on actual short runs.
comparison_rows = []
def run_variant(name, builder, epochs=EPOCHS_EXPERIMENT):
    model = builder()
    hist = model.fit(X_train, y_train, epochs=epochs,
                     validation_data=(X_valid, y_valid), verbose=0)
    best_val_acc = max(hist.history.get("val_accuracy", [float("nan")]))
    comparison_rows.append({"variant": name, "epochs_run": len(hist.history["loss"]),
                            "best_val_accuracy": best_val_acc,
                            "final_train_accuracy": hist.history["accuracy"][-1]})
    return model, hist

# The baseline history already exists; add its summary.
comparison_rows.append({
    "variant": "Phase 4 baseline (SGD/ReLU)",
    "epochs_run": len(baseline_history.history["loss"]),
    "best_val_accuracy": max(baseline_history.history["val_accuracy"]),
    "final_train_accuracy": baseline_history.history["accuracy"][-1]
})
comparison_rows.append({
    "variant": "He + ELU + BatchNorm (Adam)",
    "epochs_run": len(bn_history.history["loss"]),
    "best_val_accuracy": max(bn_history.history["val_accuracy"]),
    "final_train_accuracy": bn_history.history["accuracy"][-1]
})
pd.DataFrame(comparison_rows)

In [ ]:
# Gradient clipping worked example.
g = np.array([0.9, 100.0], dtype=float)
clip_by_value = np.clip(g, -1.0, 1.0)
norm = np.linalg.norm(g)
clip_norm = g * min(1.0, 1.0 / norm)
print("Original:", g)
print("clipvalue=1.0:", clip_by_value)
print("clipnorm=1.0:", clip_norm, "norm after clipping:", np.linalg.norm(clip_norm))
# Keras usage: keras.optimizers.SGD(learning_rate=0.01, clipvalue=1.0)
# or: keras.optimizers.SGD(learning_rate=0.01, clipnorm=1.0)

In [ ]:
# Transfer learning demonstration using the trained Fashion model as a source.
# Create a binary task from two classes and reuse lower feature layers.
source_model = baseline_model
transfer_model = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten()
])
# Copy Dense feature layers and their weights; output head is replaced.
for source_layer in source_model.layers[1:-1]:
    cloned = source_layer.__class__.from_config(source_layer.get_config())
    transfer_model.add(cloned)
    # Build layer before setting weights by running a dummy forward pass later.
transfer_model.add(keras.layers.Dense(1, activation="sigmoid"))
_ = transfer_model(np.zeros((1, 28, 28), dtype=np.float32))
for new_layer, old_layer in zip(transfer_model.layers[1:-1], source_model.layers[1:-1]):
    new_layer.set_weights(old_layer.get_weights())
for layer in transfer_model.layers[:-1]:
    layer.trainable = False

# Small two-class dataset (classes 0 and 1); labels indicate class 1.
binary_mask_train = np.isin(y_train, [0, 1])
binary_mask_valid = np.isin(y_valid, [0, 1])
X_transfer_train = X_train[binary_mask_train]
y_transfer_train = (y_train[binary_mask_train] == 1).astype("float32")
X_transfer_valid = X_valid[binary_mask_valid]
y_transfer_valid = (y_valid[binary_mask_valid] == 1).astype("float32")
transfer_model.compile(optimizer=keras.optimizers.SGD(learning_rate=1e-3),
                       loss="binary_crossentropy", metrics=["accuracy"])
transfer_history = transfer_model.fit(
    X_transfer_train, y_transfer_train, epochs=EPOCHS_EXPERIMENT,
    validation_data=(X_transfer_valid, y_transfer_valid), verbose=1
)
# Fine-tuning: unfreeze the top reused Dense layer and recompile with a lower LR.
for layer in transfer_model.layers:
    layer.trainable = True
transfer_model.compile(optimizer=keras.optimizers.SGD(learning_rate=1e-4),
                       loss="binary_crossentropy", metrics=["accuracy"])
print("Transfer-learning model ready for optional fine-tuning.")

## Phase 7 — Faster optimizers and learning-rate schedules

- Gradient Descent: \(\theta\leftarrow\theta-\eta\nabla J(\theta)\); it ignores gradient history.
- Momentum accumulates velocity and reduces oscillation; typical \(\beta=0.9\).
- Nesterov evaluates the gradient slightly ahead in the momentum direction.
- AdaGrad accumulates squared gradients, causing per-parameter learning rates to shrink; this can stop deep-network training too early.
- RMSProp uses an exponential moving average of squared gradients.
- Adam combines momentum-like first moments and RMSProp-like second moments with bias correction. Nadam adds Nesterov-style momentum.

The following cell trains five optimizer variants and plots validation loss. In quick mode, each run is intentionally short; conclusions should be based on repeated full runs, not a single seed.

In [ ]:
optimizer_factories = {
    "SGD": lambda: keras.optimizers.SGD(learning_rate=0.01),
    "Momentum": lambda: keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "Nesterov": lambda: keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True),
    "RMSProp": lambda: keras.optimizers.RMSprop(learning_rate=0.001, rho=0.9),
    "Adam": lambda: keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999)
}
optimizer_histories = {}
for name, opt_factory in optimizer_factories.items():
    model = build_bn_fashion()
    model.compile(loss="sparse_categorical_crossentropy", optimizer=opt_factory(), metrics=["accuracy"])
    hist = model.fit(X_train, y_train, epochs=EPOCHS_EXPERIMENT,
                     validation_data=(X_valid, y_valid), verbose=0)
    optimizer_histories[name] = hist.history
plt.figure(figsize=(9, 5))
for name, hist in optimizer_histories.items():
    plt.plot(hist["val_loss"], marker="o", label=name)
plt.title("Validation loss by optimizer"); plt.xlabel("Epoch"); plt.ylabel("Validation loss")
plt.legend(); plt.grid(True); plt.show()

In [ ]:
# Learning-rate schedules: constant baseline, exponential, and piecewise-constant.
def build_with_optimizer(optimizer):
    model = build_bn_fashion()
    model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
    return model

steps_per_epoch = math.ceil(len(X_train) / 256)
power_schedule = keras.optimizers.schedules.InverseTimeDecay(
    initial_learning_rate=0.01, decay_steps=max(steps_per_epoch, 1), decay_rate=1.0
)
exponential_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.01, decay_steps=max(steps_per_epoch * 2, 1), decay_rate=0.5
)
piecewise_schedule = keras.optimizers.schedules.PiecewiseConstantDecay(
    boundaries=[steps_per_epoch * 2, steps_per_epoch * 4], values=[0.01, 0.003, 0.001]
)
schedule_specs = {
    "constant": keras.optimizers.SGD(learning_rate=0.01),
    "power-like inverse-time": keras.optimizers.SGD(learning_rate=power_schedule),
    "exponential": keras.optimizers.SGD(learning_rate=exponential_schedule),
    "piecewise": keras.optimizers.SGD(learning_rate=piecewise_schedule)
}
schedule_histories = {}
for name, opt in schedule_specs.items():
    model = build_with_optimizer(opt)
    hist = model.fit(X_train, y_train, epochs=EPOCHS_EXPERIMENT,
                     validation_data=(X_valid, y_valid), verbose=0)
    schedule_histories[name] = hist.history
plt.figure(figsize=(9, 5))
for name, hist in schedule_histories.items():
    plt.plot(hist["val_loss"], marker="o", label=name)
plt.title("Learning-rate schedule comparison"); plt.xlabel("Epoch"); plt.ylabel("Validation loss")
plt.legend(); plt.grid(True); plt.show()

In [ ]:
# Learning-rate range test: short run with exponentially increasing LR.
def lr_range_test(start=1e-5, end=1.0, steps=40):
    model = build_fashion_baseline()
    rates = np.geomspace(start, end, steps)
    losses = []
    batch_size = 256
    for lr in rates:
        model.optimizer.learning_rate.assign(float(lr))
        hist = model.fit(X_train[:batch_size], y_train[:batch_size],
                         epochs=1, batch_size=64, verbose=0)
        losses.append(hist.history["loss"][-1])
    return rates, np.array(losses)

lr_rates, lr_losses = lr_range_test(steps=12 if QUICK_RUN else 40)
plt.figure(figsize=(8, 5))
plt.plot(lr_rates, lr_losses, marker=".")
plt.xscale("log"); plt.xlabel("Learning rate"); plt.ylabel("Loss")
plt.title("Learning-rate range test (illustrative, short batches)")
plt.grid(True); plt.show()
best_lr_candidate = lr_rates[np.nanargmin(lr_losses)]
print("Lowest observed loss LR (rough candidate):", best_lr_candidate)
print("A practical 1cycle schedule rises toward a tuned peak LR, then falls; momentum often moves oppositely.")

### Learning-rate schedule notes and optimizer recommendation

The task also names performance scheduling (`ReduceLROnPlateau(factor=0.5, patience=5)`) and 1cycle. 1cycle raises the learning rate toward a peak and then lowers it, often with momentum moving oppositely. A reasonable starting default for a dense tabular/image baseline is Adam with a modest learning rate and `ReduceLROnPlateau`; compare Nesterov SGD when generalization is the priority. Choose based on validation curves and repeatability.

In [ ]:
reduce_lr = keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2 if QUICK_RUN else 5)
print("Performance scheduler configured:", reduce_lr)

## Phase 8 — Regularization, MC Dropout, and capstone

Regularization options include early stopping and Batch Normalization, plus L1 (encourages sparse weights), L2 (penalizes large weights), combined L1/L2, dropout, AlphaDropout for SELU, and max-norm constraints.

Dropout randomly removes units during training; at inference ordinary dropout is off and Keras scaling keeps expected activation magnitudes aligned. If the model underfits, reduce dropout; if it overfits, increase it. **MC Dropout** leaves dropout active at prediction time and averages multiple stochastic predictions; prediction variability can act as an uncertainty signal. Avoid forcing `training=True` globally in a model containing Batch Normalization, because that would also make BN use batch statistics at inference.

In [ ]:
# functools.partial wrapper for a reusable regularized Dense layer.
RegularizedDense = partial(
    keras.layers.Dense,
    activation="elu",
    kernel_initializer="he_normal",
    kernel_regularizer=keras.regularizers.l2(0.01)
)
regularized_example = keras.Sequential([
    keras.layers.Input(shape=(20,)),
    RegularizedDense(64),
    RegularizedDense(32),
    keras.layers.Dense(1)
])
regularized_example.summary()

In [ ]:
# Compare no explicit regularizer, L2, dropout, and max-norm on Fashion MNIST.
def build_regularization_model(kind="none"):
    layers = [keras.layers.Input(shape=(28, 28)), keras.layers.Flatten()]
    if kind == "dropout":
        layers.append(keras.layers.Dropout(0.2))
    reg = keras.regularizers.l2(0.01) if kind == "l2" else None
    constraint = keras.constraints.max_norm(1.0) if kind == "max_norm" else None
    layers.append(keras.layers.Dense(300, activation="relu", kernel_regularizer=reg,
                                     kernel_constraint=constraint))
    if kind == "dropout":
        layers.append(keras.layers.Dropout(0.2))
    layers.append(keras.layers.Dense(100, activation="relu", kernel_regularizer=reg,
                                     kernel_constraint=constraint))
    if kind == "dropout":
        layers.append(keras.layers.Dropout(0.2))
    layers.append(keras.layers.Dense(10, activation="softmax"))
    model = keras.Sequential(layers)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

regularization_results = []
regularization_models = {}
for kind in ["none", "l2", "dropout", "max_norm"]:
    m = build_regularization_model(kind)
    h = m.fit(X_train, y_train, epochs=EPOCHS_EXPERIMENT,
              validation_data=(X_valid, y_valid), verbose=0)
    regularization_models[kind] = m
    regularization_results.append({
        "regularization": kind,
        "train_accuracy": h.history["accuracy"][-1],
        "validation_accuracy": h.history["val_accuracy"][-1],
        "train_val_gap": h.history["accuracy"][-1] - h.history["val_accuracy"][-1]
    })
regularization_table = pd.DataFrame(regularization_results)
regularization_table

In [ ]:
# MC Dropout model: use a dedicated layer that remains active during inference.
class MCDropout(keras.layers.Dropout):
    def call(self, inputs):
        return super().call(inputs, training=True)

def build_mc_dropout_model():
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28)),
        keras.layers.Flatten(),
        keras.layers.Dense(256, activation="elu", kernel_initializer="he_normal"),
        MCDropout(0.25),
        keras.layers.Dense(128, activation="elu", kernel_initializer="he_normal"),
        MCDropout(0.25),
        keras.layers.Dense(10, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

mc_model = build_mc_dropout_model()
mc_history = mc_model.fit(X_train, y_train, epochs=EPOCHS_EXPERIMENT,
                          validation_data=(X_valid, y_valid), verbose=1)

In [ ]:
# MC Dropout: repeat predictions and estimate uncertainty using per-class standard deviation.
N_MC = 10 if QUICK_RUN else 100
X_uncertain = X_test[:200]
mc_samples = np.stack([mc_model(X_uncertain, training=False).numpy() for _ in range(N_MC)])
# MCDropout layers force stochastic behavior even with training=False.
mc_mean = mc_samples.mean(axis=0)
mc_std = mc_samples.std(axis=0)
mc_pred = mc_mean.argmax(axis=1)
mc_confidence = mc_mean.max(axis=1)
uncertainty_score = mc_std.mean(axis=1)
mc_accuracy = accuracy_score(y_test[:len(X_uncertain)], mc_pred)
print(f"MC samples={N_MC}; subset accuracy={mc_accuracy:.3f}")
uncertainty_df = pd.DataFrame({
    "index": np.arange(len(X_uncertain)),
    "true": [class_names[i] for i in y_test[:len(X_uncertain)]],
    "predicted": [class_names[i] for i in mc_pred],
    "mean_confidence": mc_confidence,
    "mean_class_std": uncertainty_score
})
uncertainty_df.sort_values("mean_class_std", ascending=False).head(10)

In [ ]:
# Capstone model: He + ELU + BatchNorm + Dropout + Adam/Nadam + callbacks.
def build_capstone():
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28)),
        keras.layers.Flatten(),
        keras.layers.Dense(300, kernel_initializer="he_normal", use_bias=False),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("elu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(100, kernel_initializer="he_normal", use_bias=False),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("elu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(10, activation="softmax")
    ])
    model.compile(optimizer=keras.optimizers.Nadam(learning_rate=1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

capstone_model = build_capstone()
capstone_log_dir = os.path.join("logs", "task16", "capstone-" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
capstone_callbacks = [
    keras.callbacks.ModelCheckpoint("task16_capstone_best.keras", monitor="val_loss", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2 if QUICK_RUN else 8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2 if QUICK_RUN else 5),
    keras.callbacks.TensorBoard(log_dir=capstone_log_dir)
]
capstone_history = capstone_model.fit(
    X_train, y_train, epochs=EPOCHS_MAIN + 2,
    validation_data=(X_valid, y_valid), callbacks=capstone_callbacks, verbose=1
)
capstone_test_loss, capstone_test_acc = capstone_model.evaluate(X_test, y_test, verbose=0)
print(f"Capstone test accuracy: {capstone_test_acc:.4f}; loss: {capstone_test_loss:.4f}")

In [ ]:
# Ablation summary (baseline vs. capstone; ingredient-wise results above are empirical).
ablation = pd.DataFrame([
    {"configuration": "Phase 4 baseline: ReLU + SGD", "best_val_accuracy": max(baseline_history.history["val_accuracy"]),
     "ingredients": "Dense MLP; SGD"},
    {"configuration": "He + ELU + BatchNorm", "best_val_accuracy": max(bn_history.history["val_accuracy"]),
     "ingredients": "He init; ELU; BatchNorm; Adam"},
    {"configuration": "Capstone", "best_val_accuracy": max(capstone_history.history["val_accuracy"]),
     "ingredients": "He init; ELU; BatchNorm; Dropout; Nadam; LR reduction; early stopping"}
])
ablation

### Default-configuration guidance

| Hyperparameter | Practical starting point |
|---|---|
| Kernel initializer | He for ReLU/ELU; LeCun normal for SELU |
| Activation | ReLU baseline; ELU is worth testing |
| Normalization | Batch Normalization when it improves stability |
| Regularization | Early stopping first; add L2/dropout if overfitting |
| Optimizer | Adam/Nadam baseline; compare Nesterov SGD |
| Learning-rate schedule | ReduceLROnPlateau or a carefully tuned 1cycle |
| Self-normalizing alternative | Standardized inputs + LeCun normal + SELU; use AlphaDropout and avoid BatchNorm in that variant |

## Exercises (all 10 represented)

1. **TensorFlow Playground spiral:** open https://playground.tensorflow.org; try up to 3 hidden layers and 8 neurons, then compare activations/features. This browser activity cannot be automated reliably from a notebook.
2. **XOR TLU network:** use the XOR truth table in Phase 1; verify each row by hand and explain how hidden units can form intermediate regions.
3. **Output layer choice:** binary uses one sigmoid; exclusive multiclass uses softmax; multilabel uses independent sigmoid outputs.
4. **Too-high/too-low learning rate:** rerun the LR sweep and intentionally compare small/large rates; very high rates may diverge, while very low rates learn slowly.
5. **20-layer MLP:** run the optional cell below for a deep network, then compare BN and SELU variants.
6. **Initialization variance:** run the experiment below to plot pre-activation variance through 10 layers.
7. **MC Dropout N=10/50/100:** adjust `N_MC` and compare uncertainty and runtime.
8. **Transfer learning:** a two-class reused-feature experiment is included in Phase 6. For the exact exercise, adapt the masks to eight source classes and only 200 labeled examples for the remaining two.
9. **Hyperparameter search:** the SciKeras RandomizedSearchCV section covers number of layers, neurons, learning rate, and batch size; dropout can be added to the builder.
10. **Teach-back:** write up to 150 words each explaining why backprop needs the forward pass and why BatchNorm differs between training and inference.

In [ ]:
# Exercise 5: 20 hidden layers x 100 neurons, Nadam, He init (short run).
def build_deep_mlp(use_bn=True):
    layers = [keras.layers.Input(shape=(28, 28)), keras.layers.Flatten()]
    for _ in range(20):
        layers.append(keras.layers.Dense(100, kernel_initializer="he_normal", use_bias=not use_bn))
        if use_bn:
            layers.append(keras.layers.BatchNormalization())
        layers.append(keras.layers.Activation("relu"))
    layers.append(keras.layers.Dense(10, activation="softmax"))
    model = keras.Sequential(layers)
    model.compile(optimizer=keras.optimizers.Nadam(learning_rate=1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

deep_model = build_deep_mlp(use_bn=True)
deep_history = deep_model.fit(X_train, y_train, epochs=1 if QUICK_RUN else 5,
                              validation_data=(X_valid, y_valid), verbose=1)

In [ ]:
# Exercise 6: compare pre-activation variance for Normal(0,1), Glorot, and He initializers.
def activation_variances(initializer, depth=10, width=100, n_samples=512):
    x = tf.random.normal((n_samples, width), seed=SEED)
    variances = []
    for _ in range(depth):
        dense = keras.layers.Dense(width, use_bias=False, kernel_initializer=initializer)
        x = dense(x)
        variances.append(float(tf.math.reduce_variance(x).numpy()))
        x = tf.nn.relu(x)
    return variances

init_curves = {
    "Normal(0,1)": activation_variances(keras.initializers.RandomNormal(mean=0.0, stddev=1.0)),
    "Glorot": activation_variances("glorot_uniform"),
    "He": activation_variances("he_normal")
}
plt.figure(figsize=(9, 5))
for name, vals in init_curves.items():
    plt.plot(range(1, 11), vals, marker="o", label=name)
plt.xlabel("Layer"); plt.ylabel("Pre-activation variance")
plt.title("Initialization variance through 10 layers"); plt.legend(); plt.grid(True); plt.show()

## Key concepts glossary

- **TLU:** weighted sum followed by a threshold.
- **Perceptron:** single TLU layer; linear decision boundary.
- **MLP/DNN:** fully connected feedforward network with hidden layers.
- **Backpropagation:** forward computation, loss, reverse-mode gradients, parameter update.
- **Epoch:** one pass through the training set.
- **Activation:** nonlinearity after an affine layer.
- **Softplus:** `log(1 + exp(z))`, a smooth positive output activation.
- **Huber loss:** quadratic for small errors and linear for large errors.
- **Sequential / Functional / Subclassing:** Keras model-building APIs with increasing flexibility.
- **Vanishing/exploding gradients:** gradients shrink/grow through depth.
- **Glorot / He / LeCun:** variance-scaled initializers matched to activation families.
- **Batch Normalization:** mini-batch normalization followed by learned scale and shift.
- **Gradient clipping:** limit gradient values or norm.
- **Transfer learning:** reuse a pretrained model's useful layers.
- **Momentum/Nesterov/RMSProp/Adam/Nadam:** optimizers that use gradient history and/or adaptive scaling.
- **Learning-rate schedule:** change learning rate during training.
- **Dropout/MC Dropout:** regularization during training; repeated stochastic predictions for uncertainty.
- **Max-norm:** constrain incoming weight-vector norms.

## Resources
- Keras: https://keras.io/
- TensorFlow Playground: https://playground.tensorflow.org/
- Keras Tuner: https://keras.io/keras_tuner/
- TensorBoard: https://www.tensorflow.org/tensorboard
- Batch Normalization paper: https://arxiv.org/abs/1502.03167
- Adam paper: https://arxiv.org/abs/1412.6980
- MC Dropout paper: https://arxiv.org/abs/1506.02142

## Final checklist
- [ ] Phase 1: TLU/Perceptron summary, rule, trained model, XOR
- [ ] Phase 2: backprop steps, symmetry breaking, linear-collapse proof, activation plot
- [ ] Phase 3: architecture tables and four scenario recommendations
- [ ] Phase 4: Fashion MNIST model/curves/test metrics and Housing MLP vs. Task 1 baseline
- [ ] Phase 5: Wide & Deep, multi-input/output, saving/reloading, callbacks, TensorBoard, randomized search
- [ ] Phase 6: initialization/activation/BN comparison, clipping example, transfer learning
- [ ] Phase 7: five optimizers, schedules, LR sweep, recommendation
- [ ] Phase 8: regularization comparison, MC Dropout, capstone, ablation
- [ ] Exercises: all 10 attempted or documented as external/manual
- [ ] Run all cells top-to-bottom and inspect any optional dependency/dataset-download notes

**Important:** training metrics are intentionally produced by execution. They are not prefilled or invented. The supplied Task 16 README refers to a Task 1 baseline and Task 14 context that were not attached, so those comparisons require your earlier results.